# 01 — Comprensión y limpieza del PEF

Perfilado de `PEF_avance_gasto.csv`, variables derivadas y filtros analíticos.

Documentación: `data/PEF_PERFIL_Y_LIMPIEZA.md` · Pipeline: `python -m data.run_cleaning`

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data.pef_cleaning import (
    ANALYTICAL_FILTER_SPECS,
    DERIVED_COLUMN_DOCS,
    apply_analytical_filter,
    clean_pef,
    load_raw_pef,
    profile_raw,
    run_pipeline,
)
from data.paths import get_paths

sns.set_theme(style="whitegrid")
paths = get_paths()
paths

In [ ]:
raw = load_raw_pef()
perfil_entrada = profile_raw(raw)
print(f"Filas: {perfil_entrada['n_filas']:,} · Columnas: {perfil_entrada['n_columnas']}")
print("tipo_gasto:", perfil_entrada["tipo_gasto"])
pd.DataFrame(perfil_entrada["montos"]).T

In [ ]:
df = clean_pef(raw)
print("Variables derivadas:")
for k, v in DERIVED_COLUMN_DOCS.items():
    print(f"  - {k}: {v}")
df[["monto_modificado", "monto_pagado", "ratio_ejecucion", "banda_ejecucion", "sin_pago", "alta_ejecucion"]].describe(include="all")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
subset = df[df["monto_modificado"] > 0]["ratio_ejecucion"].clip(upper=2)
subset.hist(bins=50, ax=axes[0])
axes[0].set_title("ratio_ejecucion (clip 2)")
axes[0].set_xlabel("ratio")
df["banda_ejecucion"].value_counts().sort_index().plot(kind="bar", ax=axes[1])
axes[1].set_title("Bandas de ejecución")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "report" / "figures" / "01_ratio_y_bandas.png", dpi=120)
plt.show()

In [ ]:
print("Filtros analíticos:")
for name, desc in ANALYTICAL_FILTER_SPECS.items():
    n = len(apply_analytical_filter(df, name))
    print(f"  {name}: {n:,} filas — {desc}")

In [ ]:
result = run_pipeline()
print("Exportado:", result["paths"])
result["profile"]["flags"]